In [21]:
import json
from pathlib import Path
import pandas as pd

files = sorted(Path("../data").glob("reports_*.json"))
reports = pd.DataFrame([r for f in files for r in json.loads(f.read_text())])

print(len(files))
print(reports.shape)
print(reports["service_request_id"].duplicated().sum())
print(reports["requested_datetime"].min(), reports["requested_datetime"].max())

12
(20929, 17)
0
2025-10-01T00:53:24+01:00 2026-09-30T22:41:27+01:00


In [22]:
print(reports.columns.tolist())
print(reports.isna().sum())

['service_name', 'agency_responsible', 'agency_sent_datetime', 'detail', 'requested_datetime', 'updated_datetime', 'interface_used', 'service_code', 'media_url', 'title', 'description', 'long', 'status', 'lat', 'service_request_id', 'comment_count', 'requestor_name']
service_name                0
agency_responsible          0
agency_sent_datetime       41
detail                      0
requested_datetime          0
updated_datetime            0
interface_used              0
service_code                0
media_url               13128
title                       0
description                 0
long                        0
status                      0
lat                         0
service_request_id          0
comment_count            1459
requestor_name          17444
dtype: int64


In [23]:
columns = ["service_request_id", "service_name", "title", "detail", "requested_datetime", "agency_responsible", "interface_used"]
reports = reports[columns]

print(reports.shape)

(20929, 7)


In [24]:
reports["agency_responsible"] = reports["agency_responsible"].apply(lambda a: ", ".join(a.get("recipient", [])))

print(reports["agency_responsible"].nunique())
print(reports["service_name"].nunique())
print(reports["interface_used"].value_counts())

257
57
interface_used
mobile           10726
desktop           4464
Open311           2526
iOS               1381
Android            858
unknown            445
PWA (mobile)       413
PWA (desktop)       71
PWA                 45
Name: count, dtype: int64


In [25]:
reports["text"] = reports["title"].str.strip() + " " + reports["detail"].str.strip()
reports["n_words"] = reports["text"].str.split().str.len()

print(reports["n_words"].describe())
print((reports["n_words"] < 8).sum())
print(reports["text"].duplicated().sum())

count    20929.000000
mean        26.014382
std         27.484012
min          2.000000
25%          9.000000
50%         18.000000
75%         33.000000
max        624.000000
Name: n_words, dtype: float64
3804
3073


In [26]:
short = reports["n_words"] < 8
repeat = reports["text"].duplicated()

print((short & repeat).sum())
print(reports.loc[repeat, "text"].nunique())
print(reports.loc[repeat, "text"].value_counts().head(15))

2497
227
text
Dog fouling problem Dog fouling problem                                                2144
Carcass - Large problem Carcass - Large problem                                         304
Dead fox Dead fox                                                                        73
Dog fouling Dog fouling                                                                  53
Dog poo Dog poo                                                                          28
Carcass - Small problem Carcass - Small problem                                          26
Dog poo Someone hasn't cleaned up after their dog                                        25
Dead fox on pavement Dead fox on pavement                                                18
Dead animal problem Dead animal problem                                                  18
Dead badger Dead badger                                                                  15
Dead fox in road Dead fox in road                                 

In [27]:
same = reports["title"].str.strip().str.lower() == reports["detail"].str.strip().str.lower()

print(same.sum())
print((same | short | repeat).sum())
print(reports.loc[same, "interface_used"].value_counts())

3510
4953
interface_used
Open311         2521
mobile           483
desktop          330
iOS               78
Android           74
PWA (mobile)      15
unknown            9
Name: count, dtype: int64


In [28]:
keep = ~(same | short | repeat)
corpus = reports[keep].copy()

print(corpus.shape)
print(corpus["interface_used"].value_counts())
print(corpus["service_name"].value_counts().head(10))

(15976, 9)
interface_used
mobile           9451
desktop          3931
iOS              1090
Android           675
unknown           401
PWA (mobile)      315
PWA (desktop)      67
PWA                41
Open311             5
Name: count, dtype: int64
service_name
Dog fouling                                3312
Dead Animal                                2342
Dog Fouling                                2172
Dead animal                                1981
Dead animals                               1010
Dead animal (off the road)                  690
Street cleaning - Dead animal on street     599
Dog Poo/Fouling                             542
Street cleaning - Dog mess                  441
Dog Waste Clearance (Street)                393
Name: count, dtype: int64


In [29]:
import nltk

nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/Rebecca/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/Rebecca/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/Rebecca/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [30]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words("english")) | {"please", "council", "report", "reported", "problem",
                                                "outside", "near", "side", "front", "area", "need"}

def clean(text):
    tokens = word_tokenize(text.lower())
    words = [lemmatizer.lemmatize(t) for t in tokens if t.isalpha() and t not in stop_words]
    return " ".join(w for w in words if w not in stop_words)

print(clean("The dead foxes were reported to the council last week, but nobody has removed them. Please help!"))

dead fox last week nobody removed help


In [31]:
from collections import Counter

reports["clean"] = reports["text"].apply(clean)
corpus["clean"] = corpus["text"].apply(clean)

print((reports["clean"] == "").sum())
print(corpus["clean"].str.split().str.len().describe())

word_counts = Counter(" ".join(corpus["clean"]).split())
print(word_counts.most_common(30))

3
count    15976.000000
mean        16.748686
std         13.957938
min          2.000000
25%          9.000000
50%         13.000000
75%         20.000000
max        306.000000
Name: clean, dtype: float64
[('dog', 15438), ('dead', 11651), ('road', 8173), ('fox', 6548), ('poo', 4196), ('fouling', 3885), ('pavement', 3709), ('bin', 3337), ('mess', 2397), ('bag', 2030), ('street', 2023), ('waste', 1629), ('left', 1560), ('animal', 1436), ('park', 1424), ('path', 1323), ('garden', 1308), ('school', 1287), ('child', 1262), ('grass', 1206), ('house', 1170), ('large', 1160), ('cat', 1129), ('badger', 1111), ('deer', 1103), ('lane', 1080), ('along', 1067), ('owner', 1054), ('found', 1033), ('day', 1018)]


In [32]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count_vec = CountVectorizer(ngram_range=(1, 2), min_df=10)
tfidf_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=10)

X_count = count_vec.fit_transform(corpus["clean"])
X_tfidf = tfidf_vec.fit_transform(corpus["clean"])

print(X_count.shape, X_tfidf.shape)
print(count_vec.get_feature_names_out()[:20])

(15976, 4017) (15976, 4017)
['abandoned' 'abbey' 'able' 'absolute' 'absolutely' 'absolutely covered'
 'absolutely disgusting' 'absolutely stink' 'academy' 'acceptable'
 'access' 'accessed' 'accessible' 'accident' 'accidentally' 'accumulation'
 'accurate' 'acre' 'across' 'across dead']


In [33]:
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation

words = count_vec.get_feature_names_out()

def top_words(model, n=10):
    for i, topic in enumerate(model.components_):
        print(i, " ".join(words[topic.argsort()[::-1][:n]]))

k = 6
for name, X in [("BoW", X_count), ("TF-IDF", X_tfidf)]:
    lsa = TruncatedSVD(n_components=k, random_state=42).fit(X)
    lda = LatentDirichletAllocation(n_components=k, random_state=42).fit(X)
    print("\nLSA +", name)
    top_words(lsa)
    print("\nLDA +", name)
    top_words(lda)


LSA + BoW
0 dog road fouling poo dog fouling dog poo dead bin pavement mess
1 dead fox dead fox road pavement animal fox road fox dead road dead garden
2 test test test dead animal dead animal mess dog mess dog fouling dog fouling
3 poo dog poo bin bag fox overflowing poo bag dead fox poo bin waste
4 road poo dog poo badger deer dead badger pavement cat road dead animal
5 bin waste dog waste overflowing waste bin bag road bin overflowing litter emptied

LDA + BoW
0 dead fox dead fox garden found bag fox dead grass verge animal
1 road dead fox dead fox badger deer dead badger animal road dead fox road
2 dog fouling dog fouling road owner mess foul dog mess dog foul people
3 dead cat road rat park car dead cat pigeon dead rat pavement
4 dog poo dog poo bin pavement waste mess bag dog mess road
5 school child health dog issue public hazard would pavement test

LSA + TF-IDF
0 fox dead dead fox road dog fox road pavement fox dead road dead poo
1 dog poo fouling dog fouling dog poo mess dog

In [34]:
has_test = reports["clean"].str.contains(r"\btest\b")

print(has_test.sum(), (has_test & keep).sum())
print(reports.loc[has_test, "interface_used"].value_counts())
print(reports.loc[has_test, "agency_responsible"].value_counts().head(5))
print(reports.loc[has_test & keep, "clean"].head(20).tolist())


21 14
interface_used
desktop    19
mobile      1
unknown     1
Name: count, dtype: int64
agency_responsible
Royal Borough of Greenwich       13
Southwark Council                 4
Northumberland County Council     1
Cotswold District Council         1
Isle of Wight Council             1
Name: count, dtype: int64
['dog mess driving test centre alnwick previously dog mess cleared morning trailed building scrub floor obviously recurring monitoring', 'test test test test test test test test test test test test test test test test test test test testtest test test test test test test test test testtest test test test test test test test test test', 'test test test test test test test test test test test testtest test test testtest test test testtest test test testtest test test testtest test test test', 'test dead animal test dead animaltest dead animaltest dead animaltest dead animaltest dead animaltest dead animal', 'test test test test test test test test test test test test test test te

**Test reports.** The first topic comparison (k = 6) showed a topic consisting only of the word "test" (LSA + BoW). 21 reports contain "test": 20 are test submissions (mostly Royal Borough of Greenwich, via desktop), and one is a real complaint about a driving test centre. Reports in which "test" appears at least twice are therefore excluded from training and from counting.

In [35]:
is_test = reports["text"].str.lower().str.count(r"\btest\b") >= 2
keep = ~(same | short | repeat | is_test)
corpus = reports[keep].copy()

X_count = count_vec.fit_transform(corpus["clean"])
X_tfidf = tfidf_vec.fit_transform(corpus["clean"])
words = count_vec.get_feature_names_out()

print(is_test.sum())
print(corpus.shape)
print(X_count.shape, X_tfidf.shape)
print("test" in words)

20
(15963, 10)
(15963, 4015) (15963, 4015)
False


In [36]:
def compare_models(k):
    for name, X in [("BoW", X_count), ("TF-IDF", X_tfidf)]:
        lsa = TruncatedSVD(n_components=k, random_state=42).fit(X)
        lda = LatentDirichletAllocation(n_components=k, random_state=42).fit(X)
        print("\nLSA +", name)
        top_words(lsa)
        print("\nLDA +", name)
        top_words(lda)

compare_models(6)


LSA + BoW
0 dog road fouling poo dog fouling dog poo dead bin pavement mess
1 dead fox dead fox road pavement animal fox road fox dead road dead garden
2 poo dog poo bin bag fox overflowing poo bag dead fox poo bin waste
3 road poo dog poo badger deer dead badger pavement cat road dead animal
4 bin waste dog waste overflowing waste bin bag road bin overflowing litter emptied
5 mess dog mess dog fox dead fox pavement owner mess pavement mess dog dog owner

LDA + BoW
0 dead road deer animal rat verge dead deer dead animal dead rat grass
1 dead fox dead fox road cat pavement garden found fox dead fox road
2 bin dog bag poo dog poo overflowing waste litter poo bag full
3 dog poo dog poo mess dog mess road pavement fouling dog fouling owner
4 dog fouling dog fouling child pavement issue street foul school dog foul
5 road dead badger dead badger waste dog waste towards lane right hand

LSA + TF-IDF
0 fox dead dead fox road dog fox road pavement fox dead road dead poo
1 dog poo fouling dog f

In [37]:
from sklearn.metrics import silhouette_score

results = []
for k in range(4, 13, 2):
    for name, X in [("BoW", X_count), ("TF-IDF", X_tfidf)]:
        for model_name, model in [("LSA", TruncatedSVD(n_components=k, random_state=42)),
                                  ("LDA", LatentDirichletAllocation(n_components=k, random_state=42))]:
            doc_topics = model.fit_transform(X)
            labels = doc_topics.argmax(axis=1)
            score = silhouette_score(doc_topics, labels, sample_size=3000, random_state=42)
            results.append({"k": k, "model": model_name + " + " + name, "silhouette": round(score, 3)})
            print(k, model_name, name, round(score, 3))

scores = pd.DataFrame(results).pivot(index="k", columns="model", values="silhouette")
print(scores)

4 LSA BoW 0.152
4 LDA BoW 0.723
4 LSA TF-IDF 0.193
4 LDA TF-IDF 0.713
6 LSA BoW 0.105
6 LDA BoW 0.611
6 LSA TF-IDF 0.203
6 LDA TF-IDF 0.683
8 LSA BoW 0.044
8 LDA BoW 0.519
8 LSA TF-IDF 0.226
8 LDA TF-IDF 0.631
10 LSA BoW 0.01
10 LDA BoW 0.458
10 LSA TF-IDF 0.223
10 LDA TF-IDF 0.593
12 LSA BoW -0.003


KeyboardInterrupt: 

In [ ]:
compare_models(8)


LSA + BoW
0 dog road fouling poo dog fouling dog poo dead bin pavement mess
1 dead fox dead fox road pavement animal fox road fox dead road dead garden
2 poo dog poo bin bag fox overflowing poo bag dead fox poo bin waste
3 road poo dog poo badger deer dead badger pavement cat road dead animal
4 bin waste dog waste overflowing waste bin bag road bin overflowing litter emptied
5 mess dog mess dog fox dead fox pavement owner mess pavement mess dog dog owner
6 dead badger animal dead badger dead animal cat deer rat dead deer dead cat
7 pavement street bin mess dog mess fox pavement cleaning fouling school street pavement

LDA + BoW
0 dead deer animal road rat dead animal dead deer verge dead rat grass
1 dead fox dead fox road cat pavement garden fox dead found fox road
2 bin dog bag poo waste overflowing dog poo dog waste poo bag litter
3 dog poo dog poo owner road people walk footpath pavement path
4 dog fouling dog fouling issue grass owner foul sign would time
5 road dead badger dead b

**Filler words.** The topic comparisons for k = 6 and k = 8 showed positional and filler words in several topics ("towards", "right", "hand", "middle", "along", "would"). They are added to the stop word list, the texts are cleaned and vectorized again, and the four models are compared again with k = 6.

In [38]:
stop_words = stop_words | {"towards", "right", "hand", "middle", "along", "would"}

reports["clean"] = reports["text"].apply(clean)
corpus["clean"] = corpus["text"].apply(clean)

X_count = count_vec.fit_transform(corpus["clean"])
X_tfidf = tfidf_vec.fit_transform(corpus["clean"])
words = count_vec.get_feature_names_out()

print(X_count.shape)
compare_models(6)

(15963, 3954)

LSA + BoW
0 dog road fouling poo dog fouling dog poo dead bin pavement mess
1 dead fox dead fox road fox road pavement animal fox dead road dead garden
2 poo dog poo bin bag fox overflowing poo bag poo bin dead fox waste
3 road poo dog poo badger deer dead badger pavement cat road dead animal
4 bin waste dog waste overflowing waste bin bag road bin overflowing litter emptied
5 mess dog mess dog fox dead fox mess pavement pavement owner mess dog dog owner

LDA + BoW
0 dog child school fouling dog fouling owner walk issue foul path
1 dead fox dead fox road animal pavement dead animal fox road fox dead car
2 dog poo fouling dog fouling dog poo road pavement street mess dog mess
3 dead road cat badger deer dead badger verge dead deer dead cat pigeon
4 dog bin waste bag poo dog waste mess dog mess dog poo overflowing
5 dead fox dead fox bin garden rat bag dead rat black found

LSA + TF-IDF
0 fox dead dead fox road dog fox road fox dead pavement road dead poo
1 dog poo fouling

**Variant without "dog" and "dead".** As planned in the concept, LDA + BoW with k = 6 is trained once more without the two most frequent words, and compared with the baseline by top words and silhouette score.

In [39]:
variant_vec = CountVectorizer(ngram_range=(1, 2), min_df=10, stop_words=["dog", "dead"])

for name, vec in [("Baseline", count_vec), ("Without dog/dead", variant_vec)]:
    X = vec.fit_transform(corpus["clean"])
    lda = LatentDirichletAllocation(n_components=6, random_state=42)
    doc_topics = lda.fit_transform(X)
    score = silhouette_score(doc_topics, doc_topics.argmax(axis=1), sample_size=3000, random_state=42)
    print("\n" + name, X.shape, round(score, 3))
    vocab = vec.get_feature_names_out()
    for i, topic in enumerate(lda.components_):
        print(i, " ".join(vocab[topic.argsort()[::-1][:10]]))


Baseline (15963, 3954) 0.587
0 dog child school fouling dog fouling owner walk issue foul path
1 dead fox dead fox road animal pavement dead animal fox road fox dead car
2 dog poo fouling dog fouling dog poo road pavement street mess dog mess
3 dead road cat badger deer dead badger verge dead deer dead cat pigeon
4 dog bin waste bag poo dog waste mess dog mess dog poo overflowing
5 dead fox dead fox bin garden rat bag dead rat black found

Without dog/dead (15963, 3790) 0.564
0 fouling poo owner child street people pavement road walk school
1 road badger deer grass rat verge car grass verge large lane
2 bin poo bag waste overflowing left poo bag human waste bin full
3 mess road fouling pavement street footpath poo park corner st
4 fox road garden fox road pavement fox fox found bag road fox bus
5 road animal cat pavement found poo bird pigeon house number


**Final model.** LDA with Bag of Words and k = 6 gave the clearest topics. The variant without "dog" and "dead" had a lower silhouette score (0.564 vs. 0.587) and mixed dead animals with dog fouling in one topic, so the baseline is kept.

In [ ]:

final_lda = LatentDirichletAllocation(n_components=6, random_state=42)
train_topics = final_lda.fit_transform(X_count)

corpus["topic"] = train_topics.argmax(axis=1)
corpus["topic_weight"] = train_topics.max(axis=1)

top_words(final_lda, n=15)
print(corpus["topic"].value_counts().sort_index())
print(corpus.groupby("topic")["topic_weight"].median().round(2))